# Koopman Autoencoder


Libraries

In [1]:
import torch.optim
from models import Lusch
from data_generator import load_dataset, differential_dataset
from loss_functions import koopman_loss, prediction_loss
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

Define Koopman AE

In [2]:
koopman_dim = 64
hidden_dim = 500
input_dim = 3
delta_t = 0.01

epochs = 300
lr = 1e-3
Sp = 72
horizon = 72
T = max(horizon, Sp)
batch_size = 256
load_chkpt = False
chkpt_filename = "fixed_matrix_checkk"
save_every = 5
start_epoch = 1
device = "cuda"

model = Lusch(
    input_dim, koopman_dim, hidden_dim=hidden_dim, delta_t=delta_t, device=device
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

Load data

In [3]:
if load_chkpt:
    print("LOAD CHECKPOINTS")
    state_dicts = torch.load(f"{chkpt_filename}.pth")
    model.load_state_dict(state_dicts["model"])
    optimizer.load_state_dict(state_dicts["optimizer"])
    start_epoch = state_dicts["start_epoch"]
    print(start_epoch)
    print(state_dicts.keys())

X_train, X_test = load_dataset(chunk_size=1)
X_train_recon = X_train[:, :-T, :]
X_test_recon = X_test[:, :-T, :]
X_forecast_train = X_train[:, -T:, :]
X_forecast_test = X_test[:, -T:, :]
train_dl = DataLoader(differential_dataset(X_train_recon, T), batch_size=batch_size)
test_dl = DataLoader(differential_dataset(X_test_recon, T), batch_size=batch_size)

model.mu = train_dl.dataset.mu.to(device)
model.std = train_dl.dataset.std.to(device)

UnpicklingError: invalid load key, 'v'.